In [ ]:
%pip install -q fastapi uvicorn nest-asyncio pycloudflared transformers torch accelerate pydantic

import re
import threading
import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pycloudflared import try_cloudflare
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Any

# Matrix Computation Optimization on NVIDIA T4 GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

app = FastAPI()

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",  # Accelerating Long-Context Prefill
)


class PromptRequest(BaseModel):
    prompt: list | str
    system_instruction: Any = None
    max_tokens: int = 798
    temperature: float = 0.2


@app.post("/generate")
def generate_text(req: PromptRequest):
    print(f">>> [Colab] Received request from VS Code (max_tokens={req.max_tokens}, temp={req.temperature})")

    # Normaliza messages
    if isinstance(req.prompt, list):
        messages = req.prompt
    else:
        messages = []
        if req.system_instruction:
            messages.append({"role": "system", "content": req.system_instruction})
        messages.append({"role": "user", "content": str(req.prompt)})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Setting Optimal max_new_tokens for RAG Responses to Prevent Timeouts
    max_gen_tokens = min(req.max_tokens or 512, 768)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_gen_tokens,
            temperature=max(req.temperature, 0.01),
            do_sample=req.temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Strip out the entire <think>...</think> block
    clean_response = re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()
    # Handle unclosed </think> tag caused by token truncation
    if "<think>" in clean_response and "</think>" not in clean_response:
        clean_response = clean_response.split("<think>")[-1].strip()

    print(">>> [Colab] Response generation complete! Sending back to local")
    return {"response": clean_response}


nest_asyncio.apply()
public_url = try_cloudflare(port=8000)
print(f"URL For Vscode Local: {public_url.tunnel}")


def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("Server FastAPI running successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Download cloudflared...:   0%|          | 0/40274445 [00:00<?, ?it/s]

 * Running on https://shelter-notified-perception-turned.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics

URL For Vscode Local: https://shelter-notified-perception-turned.trycloudflare.com

Server FastAPI running successfully.
